# Chapter 13: In-Context Learning and Prompting

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch13_in_context_learning_and_prompting.ipynb)


## What is in this notebook, and what to change in it

Four cells, and no model is called in any of them. Chapter 13 is about what goes
into a model, so most of it is string construction, and the one cell that needs
a model's behaviour simulates it instead.

1. **The same prompt at zero shots, at three shots, and with those three
   exemplars in reverse order.** The third printing is the interesting one: the
   content is identical and only the order changed, and order is a thing
   few-shot performance is measurably sensitive to.
2. **A structured prompt**: system instruction, task, examples, output format,
   input. The same content as cell 1, arranged so each part has one job.
3. **Self-consistency by majority vote**, over simulated chain-of-thought
   answers with a 30 percent error rate. It prints one sampled chain, then the
   vote over twenty: both land on 1081, and the vote reports 65 percent
   confidence. The wrong answers are drawn from a wide range so that they
   scatter, which is the condition self-consistency depends on. If wrong answers
   clustered on one value, the vote would find that value instead, and the
   method would confidently return it.
4. **A safe arithmetic evaluator** built on the `ast` module, wired into a
   function-calling loop that computes 7392 times 4583 and gets 33877536. It
   handles arithmetic operators on literal numbers and nothing else, so unlike
   `eval` it cannot run arbitrary code. The cell's comment says to treat model
   output as untrusted, and the cell is what that looks like in practice.

Cell 3 uses the standard library's random generator, which is why this notebook
has a seed cell at the top. Without it the vote comes out differently every run.

Two edits. In cell 3, raise the error rate past 0.5 and rerun: majority voting
stops helping, and finding where it stops is more useful than the claim that it
works. In cell 4, feed the evaluator something hostile, such as
`__import__("os").system("echo hi")`, and watch it refuse. Then work out from
the code why it refuses, because that is the part you would have to get right
yourself.


> **This notebook was executed when it was built**, so the output under each
> cell is a real run's and you can read the file without running anything.
> Rebuild it with `python tools/build_notebook.py ch13`.


In [1]:
# Seeded before the chapter's own cells run.
#
# The cells below draw on random. This notebook is committed with its
# output stored, and a stored number that changes on every rebuild is
# noise printed as a result. The bundle originals are read only and
# cannot be fixed where they live, so they are seeded here instead.
#
# Same seed as tools/claim_instances.py, which produced the slide
# numbers, so a number that appears in both places appears once.
import random
random.seed(20260729)
print('seeded random with 20260729')

seeded random with 20260729


### 13.1.1 Zero-Shot, One-Shot, Few-Shot


In [2]:
def build_few_shot_prompt(exemplars, test_input, task_desc=""):
    """Construct a few-shot prompt from exemplars and test input."""
    prompt = task_desc + "\n\n" if task_desc else ""
    for inp, out in exemplars:
        prompt += f"Input: {inp}\nOutput: {out}\n\n"
    prompt += f"Input: {test_input}\nOutput:"
    return prompt

exemplars = [
    ("I love this movie!", "Positive"),
    ("Terrible waste of time.", "Negative"),
    ("A masterpiece of storytelling.", "Positive"),
]
# Zero-shot: no exemplars
print("=== Zero-shot ===")
print(build_few_shot_prompt([], "The acting was phenomenal.",
      "Classify the sentiment as Positive or Negative."))
# Few-shot: with exemplars
print("\n=== Few-shot ===")
print(build_few_shot_prompt(exemplars, "The acting was phenomenal."))
# Effect of ordering: reversed exemplars
print("\n=== Reversed order ===")
print(build_few_shot_prompt(exemplars[::-1], "The acting was phenomenal."))


=== Zero-shot ===
Classify the sentiment as Positive or Negative.

Input: The acting was phenomenal.
Output:

=== Few-shot ===
Input: I love this movie!
Output: Positive

Input: Terrible waste of time.
Output: Negative

Input: A masterpiece of storytelling.
Output: Positive

Input: The acting was phenomenal.
Output:

=== Reversed order ===
Input: A masterpiece of storytelling.
Output: Positive

Input: Terrible waste of time.
Output: Negative

Input: I love this movie!
Output: Positive

Input: The acting was phenomenal.
Output:


### 13.2.1 The Anatomy of an Effective Prompt


In [3]:
def build_structured_prompt(system, task, exemplars, user_input, fmt):
    """Build a prompt with system instruction, task, examples, and format."""
    parts = [f"System: {system}", f"Task: {task}"]
    if exemplars:
        parts.append("Examples:")
        for i, (inp, out) in enumerate(exemplars, 1):
            parts.append(f"  {i}. Input: {inp}\n     Output: {out}")
    parts.append(f"Format: {fmt}")
    parts.append(f"Input: {user_input}")
    parts.append("Output:")
    return "\n\n".join(parts)

prompt = build_structured_prompt(
    system="You are a medical triage assistant.",
    task="Classify the urgency of the patient's symptoms.",
    exemplars=[
        ("chest pain, shortness of breath",
         '{"urgency":"high","action":"call 911"}'),
        ("mild headache, no fever",
         '{"urgency":"low","action":"rest and hydrate"}')],
    user_input="fever 103F, stiff neck, sensitivity to light",
    fmt='Respond with JSON: {"urgency":"low|medium|high","action":"..."}'
)
print(prompt)


System: You are a medical triage assistant.

Task: Classify the urgency of the patient's symptoms.

Examples:

  1. Input: chest pain, shortness of breath
     Output: {"urgency":"high","action":"call 911"}

  2. Input: mild headache, no fever
     Output: {"urgency":"low","action":"rest and hydrate"}

Format: Respond with JSON: {"urgency":"low|medium|high","action":"..."}

Input: fever 103F, stiff neck, sensitivity to light

Output:


### 13.3.2 Zero-Shot CoT and Self-Consistency


In [4]:
import random
from collections import Counter

def simulate_cot_response(question, correct_answer, error_rate=0.3):
    """Simulate a CoT response with some error rate.

    Wrong answers are drawn from a broad range so that incorrect
    reasoning paths scatter their votes (the realistic setting
    self-consistency depends on), rather than clustering on the
    same few nearby values.
    """
    if random.random() > error_rate:
        return {"reasoning": "step-by-step...", "answer": correct_answer}
    wrong = correct_answer + random.randint(-20, 20)
    if wrong == correct_answer:  # ensure the noisy answer is actually wrong
        wrong += 1
    return {"reasoning": "step-by-step (with error)...", "answer": wrong}

def self_consistency(question, correct, K=20, error_rate=0.3):
    """Sample K chains and majority-vote on the final answer."""
    answers = []
    for _ in range(K):
        resp = simulate_cot_response(question, correct, error_rate)
        answers.append(resp["answer"])
    vote = Counter(answers).most_common(1)[0]
    # Return the plurality fraction (not a calibrated confidence)
    return vote[0], vote[1] / K  # answer, plurality_fraction

# Compare single chain vs self-consistency
question = "What is 23 x 47?"
correct = 1081
single = simulate_cot_response(question, correct, error_rate=0.3)
sc_answer, sc_conf = self_consistency(question, correct, K=20)
print(f"Single chain: {single['answer']} (correct={single['answer']==correct})")
print(f"Self-consistency (K=20): {sc_answer} (confidence={sc_conf:.0%})")


Single chain: 1081 (correct=True)
Self-consistency (K=20): 1081 (confidence=65%)


### 13.4.2 Function Calling: Architecture and Interface


In [5]:
import ast, json, math, operator

# Safe arithmetic evaluator: supports +, -, *, /, //, %, ** and unary
# +/- on literal numbers only. Unlike eval(), this cannot execute
# arbitrary Python code, which is essential when the expression comes
# from an LLM or user input (treat LLM output as untrusted!).
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
    ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos,
}
def safe_eval(expr):
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"unsupported expression: {ast.dump(node)}")
    return _eval(ast.parse(expr, mode="eval").body)

TOOLS = {
    "calculator": lambda expr: str(safe_eval(expr)),
    "sqrt": lambda x: str(math.sqrt(float(x))),
}

def simulate_tool_use(question):
    """Simulate the Thought -> Action -> Observation loop."""
    # Step 1: Model decides to use a tool (simulated)
    if "calculate" in question.lower() or any(c.isdigit() for c in question):
        tool_call = {"tool": "calculator", "input": "7392 * 4583"}
        print(f"Thought: I need to compute 7392 * 4583")
        print(f"Action: {json.dumps(tool_call)}")
        # Step 2: Orchestrator executes the tool
        result = TOOLS[tool_call["tool"]](tool_call["input"])
        print(f"Observation: {result}")
        # Step 3: Model incorporates result
        print(f"Answer: 7392 x 4583 = {result}")
        return result
    return "I can answer directly: ..."

simulate_tool_use("Calculate 7392 times 4583")


Thought: I need to compute 7392 * 4583
Action: {"tool": "calculator", "input": "7392 * 4583"}
Observation: 33877536
Answer: 7392 x 4583 = 33877536


'33877536'

---

## Summary

This notebook demonstrated the key code examples from Chapter 13: In-Context Learning and Prompting. For the full mathematical exposition and discussion, refer to the textbook chapter.
